# Step 4.g — SVM climate suitability, ERA5 and the four AMOC scenarios (EC-Earth3 grid, global)

For each crop, the SVM of 3.g (trained on ERA5 1981–2015 over the MIRCA2000 cropland, with the BIO variant
`CHOSEN_VARIANT` and the radiation `CHOSEN_RAD` of the tests) is applied to **every land cell**:
- **ERA5** (1981–2015): the BIO variables of 1.1.d, the reference climate;
- the **four AMOC scenarios** (2025 / 2050 stabilisation, original and hosing): the delta BIO variables of 1.3.a
  (ERA5 + EC-Earth3 change).

Suitability = the SVM probability (`predict_proba`, 0–1); binary suitability = probability ≥ `THRESHOLD` (0.5, as
before). Suitable area from the area of the EC-Earth3 cells (land cells, whole cell). Climate only: the soil
constraint comes in step 5 / 6.

Outputs in `<out_root>/4_svm_predict/mirca2000_no_threshold/<variant>_<radiation>/`: `suitability_<crop>.nc`
(`suit_proba`, `suit_bin`, dimension `scenario`), `suitable_area_<...>.csv` and maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the choices of the tests of 3.g
CHOSEN_VARIANT = None              # test 1
CHOSEN_RAD = None                  # test 2
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2015")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
THRESHOLD = 0.5
assert CHOSEN_VARIANT in ("A_60d", "B_season", "C_crop") and CHOSEN_RAD in ("rad_era5", "no_rad", "rad_fao56"), \
    "set CHOSEN_VARIANT and CHOSEN_RAD to the choices of the tests of 3.g"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_mirca2000_no_threshold", CHOSEN_VARIANT)
DELTA = cu.out_dir(CFG, "1_biovariables", "GCM_delta_mirca2000_no_threshold", SET)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_mirca2000_no_threshold", SET)
OUT = cu.out_dir(CFG, "4_svm_predict", "mirca2000_no_threshold", SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO = cu.mirca_cropland(CFG).values
# area of the EC-Earth3 cells (Mha): R^2 x dlon x (sin lat_north - sin lat_south)
R = 6371.0
dlon = np.radians(np.diff(GRID["lon_edges"]))
dsin = np.diff(np.sin(np.radians(GRID["lat_edges"])))
AREA_MHA = (R ** 2 * dsin[:, None] * dlon[None, :]) / 1e4
SCEN = {"ERA5": None, **{r: r for r in RUNS}}
print(f"set {SET} | models {MODELS}\noutputs {OUT}")

## 1. Prediction

In [ ]:
def bio_file(scenario, crop):
    if scenario == "ERA5":
        return ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
    return DELTA / f"BIOvariables_delta_{scenario}_{crop}.nc"


def predict(bundle, path):
    with xr.open_dataset(path) as ds:
        X = np.stack([ds[c].values for c in bundle["bio_cols"]], axis=-1).reshape(-1, len(bundle["bio_cols"]))
    ok = np.isfinite(X).all(axis=1) & LAND.ravel()
    p = np.full(X.shape[0], np.nan)
    p[ok] = bundle["model"].predict_proba(bundle["scaler"].transform(X[ok]))[:, 1]
    return p.reshape(LAND.shape)


rows = []
for crop in CROPS:
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    P = {s: predict(bundle, bio_file(s, crop)) for s in SCEN}
    proba = xr.DataArray(np.stack([P[s] for s in SCEN]), dims=("scenario", "lat", "lon"),
                         coords={"scenario": list(SCEN), "lat": GRID["lat"], "lon": GRID["lon"]})
    suit = xr.where(proba.notnull(), (proba >= THRESHOLD).astype("float32"), np.nan)
    xr.Dataset({"suit_proba": proba.astype("float32").assign_attrs(long_name="SVM suitability probability"),
                "suit_bin": suit.assign_attrs(long_name="binary suitability", threshold=THRESHOLD)},
               attrs={"crop": crop, "set": SET, "model": f"one-pass SVM, cost {bundle['best_cost']}, gamma {bundle['best_gamma']}",
                      "bio_cols": ", ".join(bundle["bio_cols"])}).to_netcdf(OUT / f"suitability_{crop}.nc")
    # in-sample check on the training cells (not a validation: the cross-validation of 3.g is)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    auc_in = roc_auc_score(tr.label.values, P["ERA5"][ii, jj])
    era5_area = float(np.nansum(AREA_MHA * (suit.sel(scenario="ERA5").values == 1)))
    for s in SCEN:
        b = suit.sel(scenario=s).values == 1
        a = float(np.nansum(AREA_MHA * b))
        rows.append({"crop": crop, "scenario": s, "suitable area (Mha)": a,
                     "change vs ERA5 (%)": 100 * (a - era5_area) / era5_area if s != "ERA5" else 0.0,
                     "on MIRCA cropland (Mha)": float(np.nansum(AREA_MHA * (b & CRO))),
                     "mean probability on cropland": float(np.nanmean(P[s][CRO])),
                     "in-sample AUC (ERA5)": auc_in if s == "ERA5" else np.nan})
    print(f"{crop}: ERA5 {era5_area:.1f} Mha suitable, in-sample AUC {auc_in:.3f}", flush=True)
area = pd.DataFrame(rows).set_index(["crop", "scenario"])
area.to_csv(OUT / f"suitable_area_{SET}.csv")
with pd.option_context("display.max_rows", 100, "display.width", 200):
    display(area.round(2))

## 2. Effect of the AMOC collapse: hosing − original, per stabilisation level

In [ ]:
rows = []
for crop in CROPS:
    for hos, orig in PAIRS:
        a_h, a_o = area.loc[(crop, hos), "suitable area (Mha)"], area.loc[(crop, orig), "suitable area (Mha)"]
        rows.append({"crop": crop, "pair": f"{hos} - {orig}", "area change (Mha)": a_h - a_o,
                     "area change (%)": 100 * (a_h - a_o) / a_o if a_o else np.nan})
amoc = pd.DataFrame(rows).set_index(["crop", "pair"])
amoc.to_csv(OUT / f"amoc_effect_{SET}.csv")
display(amoc.round(2))

for crop in CROPS:
    ds = xr.open_dataset(OUT / f"suitability_{crop}.nc")
    fig, axes = plt.subplots(2, 3, figsize=(20, 7))
    for ax, s in zip(axes[0], ["ERA5", PAIRS[0][1], PAIRS[1][1]]):
        ds.suit_proba.sel(scenario=s).plot(ax=ax, vmin=0, vmax=1, cmap="YlGn"); ax.set_title(f"{crop}: {s}", fontsize=9)
    axes[1, 0].axis("off")
    for ax, (hos, orig) in zip(axes[1, 1:], PAIRS):
        (ds.suit_proba.sel(scenario=hos) - ds.suit_proba.sel(scenario=orig)).plot(ax=ax, vmin=-0.5, vmax=0.5, cmap="RdBu")
        ax.set_title(f"{hos} - {orig}", fontsize=9)
    for ax in axes.ravel():
        ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout(); plt.savefig(OUT / f"suitability_maps_{SET}_{crop}.png", dpi=80); plt.show()